# 02 — Build Country-Year Feature Matrix

Assembles the master **country-year panel** used by `03_model_development` notebooks for instability prediction.
Reads cleaned parquets written by notebooks `01/01`–`01/19` and `02/01` from ADLS,
joins them on a common ISO3 key, and codes twelve binary outcome labels (all forward-shifted one year).

## Panel dimensions
- **Unit:** country-year (~167 countries × 25 years = ~4,000 rows)
- **Feature window:** 2000–2024
- **Training labels available for:** 2000–2023 (label at year *t* requires data from year *t+1*)

## Outcome labels (binary, forward-shifted +1 year)

**Core (`01_data_pull` notebooks 01–13)**

| Label | Ground truth | Coding rule |
|---|---|---|
| `civil_war_onset` | UCDP-GED | First year of state-based conflict after ≥2-year peace spell |
| `coup_attempt` | Powell-Thyne | Any coup attempt (success or failure) in year *t+1* |
| `regime_backsliding` | V-Dem | `v2x_libdem` drops ≥0.05 in year *t+1*, or regime → closed autocracy |
| `mass_unrest_onset` | ACLED | Annual protest+riot events exceed country 90th percentile in year *t+1* |
| `humanitarian_crisis_onset` | FEWS NET | Country enters IPC Phase ≥4 in year *t+1*, given Phase ≤3 at *t* |

**Supplementary (notebooks 14–19)**

| Label | Ground truth | Coding rule |
|---|---|---|
| `unrest_binary` | RSUI (14) | Annual max RSUI > country-specific 75th-percentile |
| `fh_status_decline` | Freedom House (15) | FH status worsened vs. prior year |
| `ethnic_exclusion_any` | EPR (16) | Any group >10% population coded Powerless or Discriminated |
| `epr_state_collapse` | EPR (16) | EPR codes country as State Collapse |
| `pts_high` | PTS (17) | pts_max ≥ 4 (systematic terror) |
| `pts_escalation` | PTS (17) | pts_max increased ≥1 point year-on-year |
| `aut_ep` | V-Dem ERT (18) | Country is in an active autocratization episode |

## Sources joined
Monthly sources (ACLED, GDELT, FAO food prices) are **aggregated to annual statistics**
before joining. PRIO-GRID engineered features (09b) are joined by ISO3 directly.
Supplementary sources (RSUI, Freedom House, EPR, PTS, V-Dem ERT) are
already at country-year level and merge directly.

## ADLS output
```
processed/feature_matrix/{RUN_DATE}/feature_matrix.parquet   — full feature panel
processed/feature_matrix/{RUN_DATE}/labels.parquet           — iso3 + year + 12 outcome columns
```

## Required environment variables
```
ADLS_ACCOUNT_NAME
ADLS_CONTAINER  (default: 'data')
```

In [ ]:
import os
import warnings
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from azure.identity import DefaultAzureCredential
import adlfs

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 40)

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

PANEL_START_YEAR  = 2000
PANEL_END_YEAR    = 2024
LABEL_HORIZON     = 1   # predict outcomes 1 year ahead

# Temporal split boundaries (year-inclusive)
TRAIN_END_YEAR    = 2018
VAL_END_YEAR      = 2021
# Test: 2022–2024

# ADLS prefixes for each raw source (latest date partition selected automatically)
RAW_PREFIXES = {
    # ── Core ingestion (notebooks 01–13) ──────────────────────────────────────
    "acled_monthly":    "raw/acled/monthly_agg",
    "wdi":              "raw/world_bank/wdi",
    "wgi":              "raw/world_bank/wgi",
    "vdem":             "raw/vdem",
    "polity5":          "raw/polity5",
    "ucdp_ged_cy":      "raw/ucdp_ged",
    "powell_thyne":     "raw/powell_thyne",
    "pitf":             "raw/pitf",
    "fsi":              "raw/fsi",
    "unhcr":            "raw/unhcr",
    "undp_hdi":         "raw/undp_hdi",
    "gdelt":            "raw/gdelt",
    "fao_ffpi":         "raw/fao/ffpi",
    "fao_cpi":          "raw/fao/country_cpi",
    "sipri":            "raw/sipri",
    "prio_grid_cy":     "raw/prio_grid",
    "archigos_cy":      "raw/archigos",
    "alc_cy":           "raw/alc",
    "cnts":             "raw/cnts",
    "nelda":            "raw/nelda",
    # ── PRIO-GRID engineered features (notebook 09b) ──────────────────────────
    # Reads priogrid_engineered_features.parquet from the latest raw/prio_grid
    # date partition; filename hint prevents collision with the raw yearly file.
    "priogrid_engineered": "raw/prio_grid",
    # ── Supplementary label sources (notebooks 14–19) ─────────────────────────
    "rsui":             "raw/rsui",
    "freedom_house":    "raw/freedom_house",
    "epr":              "raw/epr",
    "pts":              "raw/pts",
    "vdem_ert":         "raw/vdem_ert",
}

print(f"Run date       : {RUN_DATE}")
print(f"Panel years    : {PANEL_START_YEAR}–{PANEL_END_YEAR}")
print(f"Temporal split : train ≤{TRAIN_END_YEAR} | val ≤{VAL_END_YEAR} | test >={VAL_END_YEAR+1}")

## ADLS helpers

In [ ]:
credential     = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential":   credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

def read_latest_parquet(prefix: str, filename_hint: str = "") -> pd.DataFrame | None:
    """
    List all date-partitioned subdirectories under `prefix` and read
    the parquet from the lexicographically latest one (most recent run date).
    If `filename_hint` is provided, only files whose name contains the hint
    are read — useful when multiple parquets share the same date partition
    (e.g. raw/prio_grid holds both the yearly aggregates and the 09b
    engineered features).
    Returns None if no parquet files are found.
    """
    fs = adlfs.AzureBlobFileSystem(
        account_name=ADLS_ACCOUNT_NAME, credential=credential
    )
    full_prefix = f"{ADLS_CONTAINER}/{prefix}"
    try:
        entries = fs.ls(full_prefix, detail=False)
    except FileNotFoundError:
        print(f"  WARNING: prefix not found: {full_prefix}")
        return None

    # Keep only date-partition directories (8-digit names)
    date_dirs = sorted(
        [e for e in entries if re.search(r'/\d{8}(/|$)', e)],
        reverse=True,
    )
    if not date_dirs:
        print(f"  WARNING: no date partitions found under {full_prefix}")
        return None

    latest_dir = date_dirs[0]
    pattern = (
        f"{latest_dir}/*{filename_hint}*.parquet"
        if filename_hint
        else f"{latest_dir}/*.parquet"
    )
    parquet_files = [
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/"
        + f.replace(f"{ADLS_CONTAINER}/", "", 1)
        for f in fs.glob(pattern)
    ]
    if not parquet_files:
        print(f"  WARNING: no .parquet files matching '{pattern}'")
        return None

    dfs = [pd.read_parquet(p, storage_options=storage_options) for p in parquet_files]
    df = pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0]
    print(f"  Loaded {len(df):,} rows from {latest_dir}")
    return df

## Country crosswalk

Loads the static crosswalk from `data/country_crosswalk.csv` (committed to the repo).
Provides lookup dicts for each join-key type → ISO3.

In [ ]:
# The crosswalk CSV lives alongside the notebooks in the repo
_crosswalk_path = Path("../data/country_crosswalk.csv")
if not _crosswalk_path.exists():
    _crosswalk_path = Path("data/country_crosswalk.csv")

df_cw = pd.read_csv(_crosswalk_path, dtype=str)
df_cw["cow_numeric"]  = pd.to_numeric(df_cw["cow_numeric"],  errors="coerce")
df_cw["gw_numeric"]   = pd.to_numeric(df_cw["gw_numeric"],   errors="coerce")
df_cw["iso_numeric"]  = pd.to_numeric(df_cw["iso_numeric"],  errors="coerce")
df_cw["fews_monitored"] = df_cw["fews_monitored"].astype(int)

# Lookup dicts  →  ISO3
cow_to_iso3  = dict(zip(df_cw["cow_numeric"],  df_cw["iso3"]))
gw_to_iso3   = dict(zip(df_cw["gw_numeric"],   df_cw["iso3"]))
iso2_to_iso3 = dict(zip(df_cw["iso2"],         df_cw["iso3"]))
inum_to_iso3 = dict(zip(df_cw["iso_numeric"],  df_cw["iso3"]))
cameo_to_iso3 = dict(zip(df_cw["cameo2"],      df_cw["iso3"]))

# Name normalisation helper for string-based sources (SIPRI, FSI, UNDP)
_NAME_OVERRIDES = {
    "cote d'ivoire":              "CIV",
    "ivory coast":                "CIV",
    "iran, islamic rep.":         "IRN",
    "iran (islamic republic of)": "IRN",
    "republic of korea":          "KOR",
    "korea, rep.":                "KOR",
    "korea, south":               "KOR",
    "democratic republic of the congo": "COD",
    "congo, dem. rep.":           "COD",
    "dr congo":                   "COD",
    "congo, rep.":                "COG",
    "syrian arab republic":       "SYR",
    "bolivia (plurinational state of)": "BOL",
    "tanzania, united republic of":    "TZA",
    "united republic of tanzania":     "TZA",
    "viet nam":                   "VNM",
    "lao pdr":                    "LAO",
    "lao people's democratic republic": "LAO",
    "kyrgyz republic":            "KGZ",
    "czech republic":             "CZE",
    "slovak republic":            "SVK",
    "turkiye":                    "TUR",
    "turkey":                     "TUR",
    "egypt, arab rep.":           "EGY",
    "gambia, the":                "GMB",
    "bahamas, the":               "BHS",
    "yemen, rep.":                "YEM",
    "venezuela, rb":              "VEN",
    "micronesia, fed. sts.":      "FSM",
    "russian federation":         "RUS",
    "timor-leste":                "TLS",
    "east timor":                 "TLS",
    "cabo verde":                 "CPV",
    "eswatini":                   "SWZ",
    "swaziland":                  "SWZ",
    "north macedonia":            "MKD",
    "macedonia, former yugoslav republic of": "MKD",
}
_name_to_iso3 = dict(zip(
    df_cw["country_name_canonical"].str.lower().str.strip(),
    df_cw["iso3"]
))

def name_to_iso3(name: str) -> str | None:
    if not isinstance(name, str):
        return None
    key = re.sub(r"[^a-z0-9 ]", "", name.lower().strip())
    key_punc = name.lower().strip()
    return (
        _NAME_OVERRIDES.get(key_punc)
        or _name_to_iso3.get(key_punc)
        or _name_to_iso3.get(key)
    )

FEWS_COUNTRIES = set(df_cw.loc[df_cw["fews_monitored"] == 1, "iso3"])

print(f"Crosswalk loaded: {len(df_cw)} countries")
print(f"FEWS-monitored  : {len(FEWS_COUNTRIES)} countries")

## Panel spine

Create the rectangular country-year grid: all ISO3 codes × all years 2000–2024.
Every source is left-joined onto this spine so that absence of data produces NaN
(not a dropped row).

In [ ]:
years    = list(range(PANEL_START_YEAR, PANEL_END_YEAR + 1))
iso3s    = df_cw["iso3"].tolist()

df_spine = pd.DataFrame(
    [(iso3, yr) for iso3 in iso3s for yr in years],
    columns=["iso3", "year"],
)

print(f"Panel spine: {len(df_spine):,} country-year rows "
      f"({len(iso3s)} countries × {len(years)} years)")

## Load and join annual sources

Each annual source is read, its country identifier mapped to ISO3 via the crosswalk,
then left-joined onto the spine on `(iso3, year)`.

Sources: World Bank WDI, WGI, V-Dem, Polity5, UCDP country-year, FSI, SIPRI,
PRIO-GRID country-year aggregates, Archigos, ALC, CNTS, NELDA, UNHCR, UNDP HDI.

In [ ]:
def _add_iso3_from_cow(df, cow_col="ccode"):
    df = df.copy()
    df["iso3"] = df[cow_col].map(cow_to_iso3)
    return df

def _add_iso3_from_gw(df, gw_col="gwno"):
    df = df.copy()
    df["iso3"] = df[gw_col].map(gw_to_iso3)
    return df

def _add_iso3_from_name(df, name_col):
    df = df.copy()
    df["iso3"] = df[name_col].apply(name_to_iso3)
    return df

def _log_join(source_name, df_raw, df_joined):
    n_raw   = df_raw["iso3"].notna().sum()
    n_match = df_joined["iso3"].notna().sum()
    unmatched = df_raw[df_raw["iso3"].isna()].shape[0]
    if unmatched:
        print(f"  {source_name}: {unmatched} unmatched rows ({unmatched/len(df_raw):.1%})")

panel = df_spine.copy()

# ── World Bank WDI ────────────────────────────────────────────────────────────
raw = read_latest_parquet(RAW_PREFIXES["wdi"])
if raw is not None:
    raw.columns = [c.lower().strip() for c in raw.columns]
    eco_col  = next((c for c in raw.columns if c in ("economy", "iso3", "country_code")), None)
    yr_col   = next((c for c in raw.columns if c in ("time", "year")), None)
    if eco_col and yr_col:
        raw = raw.rename(columns={eco_col: "iso3", yr_col: "year"})
        raw["year"] = pd.to_numeric(raw["year"], errors="coerce").astype("Int64")
        drop = [c for c in raw.columns if c in ("country_name",)]
        wdi_cols = [c for c in raw.columns if c not in drop + ["iso3", "year"]]
        raw = raw[["iso3", "year"] + wdi_cols]
        panel = panel.merge(raw, on=["iso3", "year"], how="left")
        print(f"WDI: {len(wdi_cols)} feature columns added")

# ── World Bank WGI ────────────────────────────────────────────────────────────
raw = read_latest_parquet(RAW_PREFIXES["wgi"])
if raw is not None:
    raw.columns = [c.lower().strip() for c in raw.columns]
    eco_col = next((c for c in raw.columns if c in ("economy", "iso3", "country_code")), None)
    yr_col  = next((c for c in raw.columns if c in ("time", "year")), None)
    if eco_col and yr_col:
        raw = raw.rename(columns={eco_col: "iso3", yr_col: "year"})
        raw["year"] = pd.to_numeric(raw["year"], errors="coerce").astype("Int64")
        wgi_cols = [c for c in raw.columns if c not in ("iso3", "year", "country_name")]
        raw = raw[["iso3", "year"] + wgi_cols]
        panel = panel.merge(raw, on=["iso3", "year"], how="left", suffixes=("", "_wgi"))
        print(f"WGI: {len(wgi_cols)} feature columns added")

# ── V-Dem ─────────────────────────────────────────────────────────────────────
raw = read_latest_parquet(RAW_PREFIXES["vdem"])
if raw is not None:
    raw.columns = [c.lower().strip() for c in raw.columns]
    iso3_col = next((c for c in raw.columns if c in ("country_text_id", "iso3")), None)
    yr_col   = next((c for c in raw.columns if c in ("year",)), None)
    if iso3_col and yr_col:
        raw = raw.rename(columns={iso3_col: "iso3"})
        raw["year"] = pd.to_numeric(raw["year"], errors="coerce").astype("Int64")
        if "cowcode" in raw.columns:
            mask = ~raw["iso3"].isin(df_cw["iso3"])
            raw.loc[mask, "iso3"] = raw.loc[mask, "cowcode"].map(cow_to_iso3)
        vdem_cols = [c for c in raw.columns
                     if c not in ("iso3", "year", "cowcode", "country_name", "country_id")]
        raw = raw[["iso3", "year"] + vdem_cols]
        panel = panel.merge(raw, on=["iso3", "year"], how="left")
        print(f"V-Dem: {len(vdem_cols)} feature columns added")

# ── Polity5 ───────────────────────────────────────────────────────────────────
raw = read_latest_parquet(RAW_PREFIXES["polity5"])
if raw is not None:
    raw.columns = [c.lower().strip() for c in raw.columns]
    cow_col = next((c for c in raw.columns if c in ("ccode", "cow_code", "ccodecow")), None)
    yr_col  = next((c for c in raw.columns if c in ("year",)), None)
    if cow_col and yr_col:
        raw = _add_iso3_from_cow(raw, cow_col)
        raw["year"] = pd.to_numeric(raw["year"], errors="coerce").astype("Int64")
        polity_cols = [c for c in raw.columns
                       if c not in ("iso3", "year", cow_col, "country", "scode")]
        raw = raw[["iso3", "year"] + polity_cols]
        panel = panel.merge(raw, on=["iso3", "year"], how="left")
        print(f"Polity5: {len(polity_cols)} feature columns added")

print(f"\nPanel after annual sources: {panel.shape}")

In [ ]:
# ── Remaining annual sources ──────────────────────────────────────────────────

def _join_cow_source(prefix_key, feature_prefix, cow_col="ccode", yr_col="year",
                     drop_cols=(), filename_hint=""):
    raw = read_latest_parquet(RAW_PREFIXES[prefix_key], filename_hint=filename_hint)
    if raw is None:
        return
    raw.columns = [c.lower().strip() for c in raw.columns]
    ccol = next((c for c in raw.columns if c == cow_col or c in ("ccode","cow")), None)
    ycol = next((c for c in raw.columns if c in (yr_col, "year")), None)
    if not ccol or not ycol:
        print(f"  {prefix_key}: could not find COW/year columns")
        return
    raw = _add_iso3_from_cow(raw, ccol)
    raw["year"] = pd.to_numeric(raw[ycol], errors="coerce").astype("Int64")
    skip = set(drop_cols) | {ccol, ycol, "iso3", "country", "country_name",
                              "ccode", "idacr", "cow", "gwno"}
    feat_cols = [c for c in raw.columns if c not in skip]
    raw = raw.rename(columns={c: f"{feature_prefix}_{c}" for c in feat_cols})
    feat_cols_prefixed = [f"{feature_prefix}_{c}" for c in feat_cols]
    raw = raw[["iso3", "year"] + feat_cols_prefixed].drop_duplicates(["iso3","year"])
    global panel
    panel = panel.merge(raw, on=["iso3","year"], how="left")
    print(f"{prefix_key}: {len(feat_cols_prefixed)} columns added")

def _join_gw_source(prefix_key, feature_prefix, gw_col="gwno", yr_col="year",
                    drop_cols=(), filename_hint=""):
    raw = read_latest_parquet(RAW_PREFIXES[prefix_key], filename_hint=filename_hint)
    if raw is None:
        return
    raw.columns = [c.lower().strip() for c in raw.columns]
    gcol = next((c for c in raw.columns if c in (gw_col, "gwno", "gid")), None)
    ycol = next((c for c in raw.columns if c in (yr_col, "year")), None)
    if not gcol or not ycol:
        print(f"  {prefix_key}: could not find GW/year columns")
        return
    raw = _add_iso3_from_gw(raw, gcol)
    raw["year"] = pd.to_numeric(raw[ycol], errors="coerce").astype("Int64")
    skip = set(drop_cols) | {gcol, ycol, "iso3", "country", "country_name", "gwno"}
    feat_cols = [c for c in raw.columns if c not in skip]
    raw = raw.rename(columns={c: f"{feature_prefix}_{c}" for c in feat_cols})
    feat_cols_prefixed = [f"{feature_prefix}_{c}" for c in feat_cols]
    raw = raw[["iso3", "year"] + feat_cols_prefixed].drop_duplicates(["iso3","year"])
    global panel
    panel = panel.merge(raw, on=["iso3","year"], how="left")
    print(f"{prefix_key}: {len(feat_cols_prefixed)} columns added")

def _join_name_source(prefix_key, feature_prefix, name_col, yr_col="year",
                      drop_cols=(), filename_hint=""):
    raw = read_latest_parquet(RAW_PREFIXES[prefix_key], filename_hint=filename_hint)
    if raw is None:
        return
    raw.columns = [c.lower().strip() for c in raw.columns]
    ncol = next((c for c in raw.columns if c in (name_col, "country", "country_name")), None)
    ycol = next((c for c in raw.columns if c in (yr_col, "year")), None)
    if not ncol or not ycol:
        print(f"  {prefix_key}: could not find name/year columns")
        return
    raw = _add_iso3_from_name(raw, ncol)
    raw["year"] = pd.to_numeric(raw[ycol], errors="coerce").astype("Int64")
    skip = set(drop_cols) | {ncol, ycol, "iso3", "country", "country_name"}
    feat_cols = [c for c in raw.columns if c not in skip]
    raw = raw.rename(columns={c: f"{feature_prefix}_{c}" for c in feat_cols})
    feat_cols_prefixed = [f"{feature_prefix}_{c}" for c in feat_cols]
    raw = raw[["iso3", "year"] + feat_cols_prefixed].drop_duplicates(["iso3","year"])
    global panel
    panel = panel.merge(raw, on=["iso3","year"], how="left")
    print(f"{prefix_key}: {len(feat_cols_prefixed)} columns added")

# UCDP country-year aggregates (gwno)
_join_gw_source("ucdp_ged_cy",  "ucdp",     gw_col="gwno")
# Powell-Thyne coups (ccode/COW) — used later for labels; join summary cols now
_join_cow_source("powell_thyne","coup",      drop_cols=("date","day","month"))
# PITF (ccode)
_join_cow_source("pitf",        "pitf")
# FSI (country name)
_join_name_source("fsi",        "fsi",       name_col="country")
# SIPRI (country name)
_join_name_source("sipri",      "sipri",     name_col="country_name")
# PRIO-GRID country-year aggregates (gwno).
# filename_hint="yearly" scopes to the raw yearly aggregate file and prevents
# accidental concatenation with priogrid_engineered_features.parquet when both
# land in the same date partition (i.e. notebooks 09 and 09b ran on the same day).
_join_gw_source("prio_grid_cy", "prio",      gw_col="gwno", filename_hint="yearly")
# Archigos (ccode)
_join_cow_source("archigos_cy", "arch",      drop_cols=("idacr",))
# ALC — Africa only; ccode
_join_cow_source("alc_cy",      "alc",       drop_cols=("country",))
# CNTS (ccode)
_join_cow_source("cnts",        "cnts",      drop_cols=("country",))
# NELDA (cow)
_join_cow_source("nelda",       "nelda",     cow_col="cow", drop_cols=("country",))
# UNHCR (iso3 direct)
raw = read_latest_parquet(RAW_PREFIXES["unhcr"])
if raw is not None:
    raw.columns = [c.lower().strip() for c in raw.columns]
    yr_col = next((c for c in raw.columns if c in ("year",)), None)
    iso_col = next((c for c in raw.columns if c in ("iso3","iso","country_of_asylum_iso")), None)
    if iso_col and yr_col:
        raw = raw.rename(columns={iso_col: "iso3"})
        raw["year"] = pd.to_numeric(raw[yr_col], errors="coerce").astype("Int64")
        feat_cols = [c for c in raw.columns if c not in ("iso3","year","country_name")]
        raw = raw.rename(columns={c: f"unhcr_{c}" for c in feat_cols})
        panel = panel.merge(raw[["iso3","year"]+[f"unhcr_{c}" for c in feat_cols]],
                            on=["iso3","year"], how="left")
        print(f"UNHCR: {len(feat_cols)} columns added")
# UNDP HDI (country name)
_join_name_source("undp_hdi",   "hdi",       name_col="country")
# FAO country CPI (country name or iso3)
_join_name_source("fao_cpi",    "fao_cpi",   name_col="country")

print(f"\nPanel after all annual sources: {panel.shape}")

## Aggregate monthly sources to annual and join

ACLED, GDELT, and FAO food price index are monthly. We compute annual summary
statistics (sum, mean, std) and join them onto the country-year panel.

In [ ]:
# ── ACLED monthly → annual ────────────────────────────────────────────────────
raw_acled = read_latest_parquet(RAW_PREFIXES["acled_monthly"])
if raw_acled is not None:
    raw_acled.columns = [c.lower().strip() for c in raw_acled.columns]

    # Resolve country identifier to ISO3
    iso_col = next((c for c in raw_acled.columns
                    if c in ("iso3", "iso", "country_iso")), None)
    if iso_col == "iso" and raw_acled["iso"].dtype in (int, float, "Int64"):
        # ISO numeric integer
        raw_acled["iso3"] = raw_acled["iso"].map(inum_to_iso3)
    elif iso_col:
        raw_acled = raw_acled.rename(columns={iso_col: "iso3"})
    else:
        name_col = next((c for c in raw_acled.columns if "country" in c), None)
        if name_col:
            raw_acled["iso3"] = raw_acled[name_col].apply(name_to_iso3)

    # Extract year from year_month ("YYYY-MM") or year column
    if "year" not in raw_acled.columns:
        ym_col = next((c for c in raw_acled.columns if "year_month" in c or c == "ym"), None)
        if ym_col:
            raw_acled["year"] = raw_acled[ym_col].str[:4].astype(int)

    count_cols = [c for c in raw_acled.columns
                  if c.startswith("events_") or c.startswith("fatalities")]

    if "iso3" in raw_acled.columns and "year" in raw_acled.columns and count_cols:
        acled_agg = (
            raw_acled[["iso3", "year"] + count_cols]
            .groupby(["iso3", "year"], as_index=False)
            .agg(
                **{f"acled_{c}_annual":    (c, "sum") for c in count_cols},
                **{f"acled_{c}_mean_mo":   (c, "mean") for c in count_cols},
                **{f"acled_{c}_std_mo":    (c, "std")  for c in count_cols},
            )
        )
        acled_agg["year"] = acled_agg["year"].astype("Int64")
        panel = panel.merge(acled_agg, on=["iso3", "year"], how="left")
        print(f"ACLED: {len(acled_agg.columns)-2} annual aggregate columns added")

# ── GDELT monthly → annual ────────────────────────────────────────────────────
raw_gdelt = read_latest_parquet(RAW_PREFIXES["gdelt"])
if raw_gdelt is not None:
    raw_gdelt.columns = [c.lower().strip() for c in raw_gdelt.columns]
    raw_gdelt["iso3"] = raw_gdelt["cameo_country"].map(cameo_to_iso3)

    if "year" not in raw_gdelt.columns:
        ym_col = next((c for c in raw_gdelt.columns if "year_month" in c), None)
        if ym_col:
            raw_gdelt["year"] = raw_gdelt[ym_col].str[:4].astype(int)

    gdelt_numeric = [c for c in raw_gdelt.columns
                     if c not in ("iso3", "year", "cameo_country", "year_month",
                                  "country_label", "cameo_country")
                     and raw_gdelt[c].dtype in ("float64", "int64", "Int64")]

    if "iso3" in raw_gdelt.columns and "year" in raw_gdelt.columns and gdelt_numeric:
        gdelt_agg_spec = {}
        for c in gdelt_numeric:
            if "count" in c or c.startswith("events_") or c == "mentions_total":
                gdelt_agg_spec[f"gdelt_{c}_annual"] = (c, "sum")
            else:
                gdelt_agg_spec[f"gdelt_{c}_mean"] = (c, "mean")
                gdelt_agg_spec[f"gdelt_{c}_std"]  = (c, "std")

        gdelt_agg = (
            raw_gdelt[["iso3", "year"] + gdelt_numeric]
            .groupby(["iso3", "year"], as_index=False)
            .agg(**gdelt_agg_spec)
        )
        gdelt_agg["year"] = gdelt_agg["year"].astype("Int64")
        panel = panel.merge(gdelt_agg, on=["iso3", "year"], how="left")
        print(f"GDELT: {len(gdelt_agg.columns)-2} annual aggregate columns added")

# ── FAO Food Price Index → annual mean ───────────────────────────────────────
raw_ffpi = read_latest_parquet(RAW_PREFIXES["fao_ffpi"])
if raw_ffpi is not None:
    raw_ffpi.columns = [c.lower().strip() for c in raw_ffpi.columns]
    # FFPI is a global series (no country column) — broadcast to all countries
    val_col  = next((c for c in raw_ffpi.columns if "index_value" in c or c == "value"), None)
    name_col = next((c for c in raw_ffpi.columns if "index_name" in c or c == "name"), None)
    yr_col   = next((c for c in raw_ffpi.columns if c == "year"), None)

    if val_col and name_col and yr_col:
        ffpi_wide = (
            raw_ffpi.groupby([yr_col, name_col])[val_col].mean()
            .unstack(name_col)
            .reset_index()
            .rename(columns={yr_col: "year"})
        )
        ffpi_wide.columns = ["year"] + [
            f"ffpi_{c.lower().replace(' ','_')}" for c in ffpi_wide.columns[1:]
        ]
        ffpi_wide["year"] = pd.to_numeric(ffpi_wide["year"], errors="coerce").astype("Int64")
        # Broadcast: cross-join with all iso3s via spine year
        panel = panel.merge(ffpi_wide, on="year", how="left")
        print(f"FAO FFPI: {len(ffpi_wide.columns)-1} global price index columns added")

print(f"\nPanel after monthly sources: {panel.shape}")

## Join PRIO-GRID engineered features (notebook `02/01`)

`01_prio_grid_spatial_features` writes a country-year parquet containing eight
feature families derived from grid-cell-level spatial analysis:

| Method | Features (prefix `prio_`) |
|---|---|
| 1 — Distributional aggregates | `ntl_mean/std/p10/p90/p90p10_ratio`, `pop_sum/std/top10cell_share` |
| 2 — NTL Gini / darkness | `ntl_gini`, `ntl_darkness_frac`, `ntl_urban_frac` |
| 3 — Pop-weighted climate stress | `pop_wtd_rainfall_sd`, `pop_wtd_temp_sd`, `high_stress_pop_frac` |
| 4 — Conflict cell density | `conflict_cell_count/frac`, `conflict_pop_frac`, `conflict_deaths_per_cell` |
| 5 — Lootable resource exposure | `petroleum_cell/pop_frac`, `drug_cell/pop_frac`, `resource_any_pop_frac` |
| 6 — Core/periphery polarization | `ntl_core_mean`, `ntl_periphery_mean`, `ntl_core_periphery_ratio`, `ntl_cv` |
| 7 — NTL economic shock | `ntl_popwtd_mean`, `ntl_yoy_change`, `ntl_shock_zscore`, `ntl_negative_shock` |
| 8 — GeoEPR ethnic territory | `geoepr_excl/incl_ntl_mean`, `geoepr_ntl_excl_gap`, `geoepr_excl_pop_share`, `geoepr_ttime_excl_gap`, `geoepr_n_excl_groups` |

These are keyed by `(iso3, year)` and merge directly onto the panel. The
`filename_hint="engineered_features"` ensures we read only the 09b output and not
the raw yearly aggregate written by notebook `01/09`, even if both land in the same
date partition.

In [ ]:
# ── PRIO-GRID engineered features (notebook 09b) ──────────────────────────────
raw_prio_eng = read_latest_parquet(
    RAW_PREFIXES["priogrid_engineered"],
    filename_hint="engineered_features",
)

if raw_prio_eng is not None:
    raw_prio_eng.columns = [c.lower().strip() for c in raw_prio_eng.columns]
    raw_prio_eng["year"] = pd.to_numeric(raw_prio_eng["year"], errors="coerce").astype("Int64")

    eng_cols = [c for c in raw_prio_eng.columns if c not in ("iso3", "year")]
    raw_prio_eng = raw_prio_eng[["iso3", "year"] + eng_cols].drop_duplicates(["iso3", "year"])

    panel = panel.merge(raw_prio_eng, on=["iso3", "year"], how="left")
    print(f"PRIO-GRID engineered (09b): {len(eng_cols)} feature columns added")
    print(f"  Features: {eng_cols}")
else:
    print("PRIO-GRID engineered (09b): not found — run notebook 09b first")

print(f"\nPanel after PRIO-GRID engineered features: {panel.shape}")

## Join supplementary label sources (notebooks 14–19)

RSUI (14), Freedom House (15), EPR (16), PTS (17), V-Dem ERT (18), and Cline coup (19)
are all already at the country-year level with ISO3 keys — they merge directly onto the
panel without any aggregation. Each source contributes both predictor columns and the
pre-computed binary flags used as outcome labels in the next section.

In [ ]:
def _join_iso3_source(prefix_key, drop_cols=()):
    """Join a parquet that is already keyed by (iso3, year) onto the panel."""
    raw = read_latest_parquet(RAW_PREFIXES[prefix_key])
    if raw is None:
        print(f"  {prefix_key}: not found — skipping")
        return
    raw.columns = [c.lower().strip() for c in raw.columns]
    raw["year"] = pd.to_numeric(raw["year"], errors="coerce").astype("Int64")
    skip = set(drop_cols) | {"country_name", "country"}
    feat_cols = [c for c in raw.columns if c not in skip]
    raw = raw[feat_cols].drop_duplicates(["iso3", "year"])
    global panel
    panel = panel.merge(raw, on=["iso3", "year"], how="left")
    added = len(feat_cols) - 2  # subtract iso3 + year
    print(f"{prefix_key}: {added} columns added")


# ── 14 RSUI — social unrest index (monthly → annual already done in nb14) ─────
# Predictors: rsui_mean, rsui_max, rsui_std, rsui_*_mean/max
# Label col : unrest_binary
_join_iso3_source("rsui", drop_cols=("rsui_monthly",))

# ── 15 Freedom House ──────────────────────────────────────────────────────────
# Predictors: fh_pr, fh_cl, fh_total, fh_status_ord
# Label cols: fh_status_decline, fh_pr_decline
_join_iso3_source("freedom_house", drop_cols=("fh_status",))

# ── 16 EPR-Core (Ethnic Power Relations) ──────────────────────────────────────
# Predictors: n_excluded_groups, excluded_pop_share, n_relevant_groups,
#             max_group_size, dominant_group_flag
# Label cols: ethnic_exclusion_any, epr_state_collapse
_join_iso3_source("epr")

# ── 17 Political Terror Scale ─────────────────────────────────────────────────
# Predictors: pts_a, pts_s, pts_h, pts_mean, pts_max
# Label cols: pts_high, pts_escalation
_join_iso3_source("pts")

# ── 18 V-Dem ERT (autocratization episodes) ───────────────────────────────────
# Predictors: aut_ep_duration, dem_ep, dem_ep_start_year, edi, edi_change_3y
# Label col : aut_ep
_join_iso3_source("vdem_ert", drop_cols=("reg_trans_type", "aut_ep_outcome"))

print(f"\nPanel after supplementary sources: {panel.shape}")

## Outcome label coding

Twelve binary outcomes, each forward-shifted +1 year so that the label in row
`(iso3, year)` represents what happens in year `t+1`.

**Core outcomes (notebooks 01–13)**

| Label | Source | Coding rule |
|---|---|---|
| `civil_war_onset` | UCDP-GED | First year of state-based conflict after ≥2-year peace spell |
| `coup_attempt` | Powell-Thyne | Any coup attempt (success or failure) |
| `regime_backsliding` | V-Dem `v2x_libdem`, `v2x_regime` | libdem drop ≥0.05 OR regime → closed autocracy |
| `mass_unrest_onset` | ACLED | Annual protest+riot events exceed country 90th-percentile (train years only) |
| `humanitarian_crisis_onset` | FEWS NET | IPC phase ≤3 → ≥4 (~39 FEWS countries; NA elsewhere) |

**Supplementary outcomes (notebooks 14–18)**

| Label | Source | Coding rule |
|---|---|---|
| `unrest_binary` | RSUI (14) | Annual max RSUI > country-specific 75th-percentile |
| `fh_status_decline` | Freedom House (15) | FH status category worsened vs. prior year |
| `ethnic_exclusion_any` | EPR (16) | Any group >10% population coded Powerless or Discriminated |
| `epr_state_collapse` | EPR (16) | EPR codes country as State Collapse |
| `pts_high` | PTS (17) | pts_max ≥ 4 (systematic terror) |
| `pts_escalation` | PTS (17) | pts_max increased ≥1 point year-on-year |
| `aut_ep` | V-Dem ERT (18) | Country is in an active autocratization episode |

In [ ]:
# ── Helper: shift a country-level series forward by 1 year ────────────────────
def _shift_label(df_panel: pd.DataFrame, col: str) -> pd.Series:
    """
    For each (iso3, year) row, return the value of `col` from year+1.
    Uses a sorted merge rather than groupby.shift so that gaps in year
    sequences are handled correctly.
    """
    ref = df_panel[["iso3", "year", col]].copy()
    ref = ref.rename(columns={"year": "year_next", col: col + "_next"})
    ref["year"] = ref["year_next"] - 1
    merged = df_panel[["iso3", "year"]].merge(
        ref[["iso3", "year", col + "_next"]], on=["iso3", "year"], how="left"
    )
    return merged[col + "_next"]

labels = pd.DataFrame({"iso3": panel["iso3"], "year": panel["year"]})

# ─────────────────────────────────────────────────────────────────────────────
# 1. civil_war_onset
#    Source: UCDP. Column `ucdp_conflict_new_battle` (or similar) flags whether
#    a new state-based armed conflict (≥25 battle deaths) began in that year.
#    We additionally require ≥2 consecutive conflict-free years before onset.
# ─────────────────────────────────────────────────────────────────────────────
ucdp_conflict_col = next(
    (c for c in panel.columns if "new_conflict" in c or "conflict_onset" in c
     or "new_battle" in c or c == "ucdp_conflict_new_battle"),
    None,
)
ucdp_active_col = next(
    (c for c in panel.columns if "ucdp" in c and ("active" in c or "conflict_year" in c)),
    None,
)

if ucdp_conflict_col:
    labels["civil_war_onset"] = _shift_label(panel, ucdp_conflict_col).fillna(0).astype("Int8")
    print(f"civil_war_onset: coded from '{ucdp_conflict_col}' | base rate = "
          f"{labels['civil_war_onset'].mean():.3f}")
elif ucdp_active_col:
    tmp = panel[["iso3", "year", ucdp_active_col]].copy()
    tmp = tmp.sort_values(["iso3", "year"])
    tmp["_prev1"] = tmp.groupby("iso3")[ucdp_active_col].shift(1)
    tmp["_prev2"] = tmp.groupby("iso3")[ucdp_active_col].shift(2)
    tmp["_peace_spell"] = ((tmp["_prev1"] == 0) & (tmp["_prev2"] == 0)).astype(int)
    tmp["_onset_raw"] = ((tmp[ucdp_active_col] == 1) & (tmp["_peace_spell"] == 1)).astype(int)
    onset_col_name = "civil_war_onset_raw"
    panel[onset_col_name] = tmp["_onset_raw"].values
    labels["civil_war_onset"] = _shift_label(panel, onset_col_name).fillna(0).astype("Int8")
    panel.drop(columns=[onset_col_name], inplace=True)
    print(f"civil_war_onset: derived from '{ucdp_active_col}' + peace spell | "
          f"base rate = {labels['civil_war_onset'].mean():.3f}")
else:
    labels["civil_war_onset"] = pd.array([pd.NA] * len(labels), dtype="Int8")
    print("civil_war_onset: UCDP columns not found — label is NA")

# ─────────────────────────────────────────────────────────────────────────────
# 2. coup_attempt
#    Source: Powell-Thyne. Column `coup_any_attempt` (or similar) = 1 if any
#    coup attempt (success or failure) occurred in that country-year.
# ─────────────────────────────────────────────────────────────────────────────
coup_col = next(
    (c for c in panel.columns
     if ("coup" in c and ("attempt" in c or "any" in c)) or c == "coup_ptcoup_any"),
    None,
)
if coup_col:
    labels["coup_attempt"] = _shift_label(panel, coup_col).fillna(0).astype("Int8")
    print(f"coup_attempt   : coded from '{coup_col}' | base rate = "
          f"{labels['coup_attempt'].mean():.3f}")
else:
    labels["coup_attempt"] = pd.array([pd.NA] * len(labels), dtype="Int8")
    print("coup_attempt   : Powell-Thyne columns not found — label is NA")

# ─────────────────────────────────────────────────────────────────────────────
# 3. regime_backsliding
#    Source: V-Dem. Two conditions (OR):
#      a) v2x_libdem drops ≥0.05 from year t to year t+1
#      b) v2x_regime transitions from ≥2 (electoral democracy) to 0 (closed autocracy)
# ─────────────────────────────────────────────────────────────────────────────
libdem_col = next(
    (c for c in panel.columns if "v2x_libdem" in c or c == "vdem_v2x_libdem"),
    None,
)
regime_col = next(
    (c for c in panel.columns if "v2x_regime" in c or c == "vdem_v2x_regime"),
    None,
)

if libdem_col or regime_col:
    cond_a = pd.Series(False, index=panel.index)
    cond_b = pd.Series(False, index=panel.index)
    if libdem_col:
        libdem_next = _shift_label(panel, libdem_col)
        cond_a = (panel[libdem_col] - libdem_next >= 0.05)
    if regime_col:
        regime_next = _shift_label(panel, regime_col)
        cond_b = ((panel[regime_col] >= 2) & (regime_next == 0))
    labels["regime_backsliding"] = (cond_a | cond_b).astype("Int8")
    print(f"regime_backsliding: coded | base rate = "
          f"{labels['regime_backsliding'].mean():.3f}")
else:
    labels["regime_backsliding"] = pd.array([pd.NA] * len(labels), dtype="Int8")
    print("regime_backsliding: V-Dem columns not found — label is NA")

# ─────────────────────────────────────────────────────────────────────────────
# 4. mass_unrest_onset
#    Source: ACLED annual aggregate. Onset = annual protest+riot event count in
#    year t+1 exceeds the country's 90th-percentile count (training years only).
# ─────────────────────────────────────────────────────────────────────────────
protest_col = next(
    (c for c in panel.columns if "protest" in c and "annual" in c and "acled" in c),
    None,
)
if protest_col is None:
    protest_col = next(
        (c for c in panel.columns if "acled" in c and "annual" in c), None
    )

if protest_col:
    train_mask = panel["year"] <= TRAIN_END_YEAR
    q90 = (
        panel.loc[train_mask, ["iso3", protest_col]]
        .groupby("iso3")[protest_col].quantile(0.9).rename("q90")
    )
    tmp = panel[["iso3", "year", protest_col]].merge(q90, on="iso3", how="left")
    panel["_unrest_flag"] = (tmp[protest_col] > tmp["q90"]).astype(int).values
    labels["mass_unrest_onset"] = _shift_label(panel, "_unrest_flag").fillna(0).astype("Int8")
    panel.drop(columns=["_unrest_flag"], inplace=True)
    print(f"mass_unrest_onset  : coded from '{protest_col}' (>country Q90) | "
          f"base rate = {labels['mass_unrest_onset'].mean():.3f}")
else:
    labels["mass_unrest_onset"] = pd.array([pd.NA] * len(labels), dtype="Int8")
    print("mass_unrest_onset  : ACLED protest columns not found — label is NA")

# ─────────────────────────────────────────────────────────────────────────────
# 5. humanitarian_crisis_onset
#    Source: FEWS NET IPC phase. Onset = IPC ≤3 at t → ≥4 at t+1.
#    Only coded for FEWS-monitored countries; NA elsewhere.
# ─────────────────────────────────────────────────────────────────────────────
fews_col = next(
    (c for c in panel.columns if "fews" in c and ("ipc" in c or "phase" in c)),
    None,
)
if fews_col:
    fews_next = _shift_label(panel, fews_col)
    onset = (
        panel["iso3"].isin(FEWS_COUNTRIES)
        & (panel[fews_col] <= 3)
        & (fews_next >= 4)
    ).astype("Int8")
    onset[~panel["iso3"].isin(FEWS_COUNTRIES)] = pd.NA
    labels["humanitarian_crisis_onset"] = pd.array(onset, dtype="Int8")
    print(f"humanitarian_crisis_onset: coded from '{fews_col}' | "
          f"base rate (FEWS) = {onset[panel['iso3'].isin(FEWS_COUNTRIES)].mean():.3f}")
else:
    raw_fews = read_latest_parquet("raw/fews_net")
    if raw_fews is not None:
        raw_fews.columns = [c.lower().strip() for c in raw_fews.columns]
        ipc_col = next((c for c in raw_fews.columns if "ipc" in c or "phase" in c), None)
        iso_col = next((c for c in raw_fews.columns if c in ("iso3", "iso")), None)
        yr_col  = next((c for c in raw_fews.columns if c == "year"), None)
        if ipc_col and iso_col and yr_col:
            fews_cy = (
                raw_fews.rename(columns={iso_col: "iso3", yr_col: "year"})
                [["iso3", "year", ipc_col]]
                .groupby(["iso3", "year"])[ipc_col].max()
                .reset_index(name="fews_max_ipc")
            )
            panel = panel.merge(fews_cy, on=["iso3", "year"], how="left")
            fews_next = _shift_label(panel, "fews_max_ipc")
            onset = (
                panel["iso3"].isin(FEWS_COUNTRIES)
                & (panel["fews_max_ipc"] <= 3)
                & (fews_next >= 4)
            ).astype("Int8")
            onset[~panel["iso3"].isin(FEWS_COUNTRIES)] = pd.NA
            labels["humanitarian_crisis_onset"] = pd.array(onset, dtype="Int8")
            print("humanitarian_crisis_onset: coded from FEWS raw (late load)")
    else:
        labels["humanitarian_crisis_onset"] = pd.array([pd.NA] * len(labels), dtype="Int8")
        print("humanitarian_crisis_onset: FEWS data not found — label is NA")

# ─────────────────────────────────────────────────────────────────────────────
# 6. unrest_binary   (nb14 — RSUI)
#    Pre-computed: 1 if annual max RSUI > country's historical 75th percentile.
# ─────────────────────────────────────────────────────────────────────────────
if "unrest_binary" in panel.columns:
    labels["unrest_binary"] = _shift_label(panel, "unrest_binary").fillna(0).astype("Int8")
    print(f"unrest_binary      : coded from RSUI | base rate = "
          f"{labels['unrest_binary'].mean():.3f}")
else:
    labels["unrest_binary"] = pd.array([pd.NA] * len(labels), dtype="Int8")
    print("unrest_binary      : RSUI not joined — label is NA")

# ─────────────────────────────────────────────────────────────────────────────
# 7. fh_status_decline   (nb15 — Freedom House)
#    Pre-computed: 1 if Freedom House status category worsened vs. prior year.
# ─────────────────────────────────────────────────────────────────────────────
if "fh_status_decline" in panel.columns:
    labels["fh_status_decline"] = (
        _shift_label(panel, "fh_status_decline").fillna(0).astype("Int8")
    )
    print(f"fh_status_decline  : coded from Freedom House | base rate = "
          f"{labels['fh_status_decline'].mean():.3f}")
else:
    labels["fh_status_decline"] = pd.array([pd.NA] * len(labels), dtype="Int8")
    print("fh_status_decline  : Freedom House not joined — label is NA")

# ─────────────────────────────────────────────────────────────────────────────
# 8. ethnic_exclusion_any   (nb16 — EPR-Core)
#    Pre-computed: 1 if any group >10% of population is Powerless or Discriminated.
# ─────────────────────────────────────────────────────────────────────────────
if "ethnic_exclusion_any" in panel.columns:
    labels["ethnic_exclusion_any"] = (
        _shift_label(panel, "ethnic_exclusion_any").fillna(0).astype("Int8")
    )
    print(f"ethnic_exclusion_any: coded from EPR | base rate = "
          f"{labels['ethnic_exclusion_any'].mean():.3f}")
else:
    labels["ethnic_exclusion_any"] = pd.array([pd.NA] * len(labels), dtype="Int8")
    print("ethnic_exclusion_any: EPR not joined — label is NA")

# ─────────────────────────────────────────────────────────────────────────────
# 9. epr_state_collapse   (nb16 — EPR-Core)
#    Pre-computed: 1 if EPR codes country as State Collapse this year.
# ─────────────────────────────────────────────────────────────────────────────
if "epr_state_collapse" in panel.columns:
    labels["epr_state_collapse"] = (
        _shift_label(panel, "epr_state_collapse").fillna(0).astype("Int8")
    )
    print(f"epr_state_collapse  : coded from EPR | base rate = "
          f"{labels['epr_state_collapse'].mean():.3f}")
else:
    labels["epr_state_collapse"] = pd.array([pd.NA] * len(labels), dtype="Int8")
    print("epr_state_collapse  : EPR not joined — label is NA")

# ─────────────────────────────────────────────────────────────────────────────
# 10. pts_high   (nb17 — Political Terror Scale)
#     Pre-computed: 1 if pts_max ≥ 4 (systematic terror affecting large populations).
# ─────────────────────────────────────────────────────────────────────────────
if "pts_high" in panel.columns:
    labels["pts_high"] = _shift_label(panel, "pts_high").fillna(0).astype("Int8")
    print(f"pts_high           : coded from PTS | base rate = "
          f"{labels['pts_high'].mean():.3f}")
else:
    labels["pts_high"] = pd.array([pd.NA] * len(labels), dtype="Int8")
    print("pts_high           : PTS not joined — label is NA")

# ─────────────────────────────────────────────────────────────────────────────
# 11. pts_escalation   (nb17 — Political Terror Scale)
#     Pre-computed: 1 if pts_max increased ≥1 point vs. prior year.
# ─────────────────────────────────────────────────────────────────────────────
if "pts_escalation" in panel.columns:
    labels["pts_escalation"] = _shift_label(panel, "pts_escalation").fillna(0).astype("Int8")
    print(f"pts_escalation     : coded from PTS | base rate = "
          f"{labels['pts_escalation'].mean():.3f}")
else:
    labels["pts_escalation"] = pd.array([pd.NA] * len(labels), dtype="Int8")
    print("pts_escalation     : PTS not joined — label is NA")

# ─────────────────────────────────────────────────────────────────────────────
# 12. aut_ep   (nb18 — V-Dem ERT)
#     Pre-computed: 1 if country is in an active autocratization episode.
# ─────────────────────────────────────────────────────────────────────────────
if "aut_ep" in panel.columns:
    labels["aut_ep"] = _shift_label(panel, "aut_ep").fillna(0).astype("Int8")
    print(f"aut_ep             : coded from V-Dem ERT | base rate = "
          f"{labels['aut_ep'].mean():.3f}")
else:
    labels["aut_ep"] = pd.array([pd.NA] * len(labels), dtype="Int8")
    print("aut_ep             : V-Dem ERT not joined — label is NA")

print("\nLabel summary:")
print(labels.drop(columns=["iso3", "year"]).describe())

## Lag features and missingness flags

For each numeric feature column:
- Annual lags: t-1, t-2
- Rolling means: 3-year and 5-year (lagged ≥1, no leakage)
- Binary missingness flag for columns with >5% missing

In [ ]:
OUTCOME_COLS = [
    # Core outcomes (notebooks 01–13 sources)
    "civil_war_onset",
    "coup_attempt",
    "regime_backsliding",
    "mass_unrest_onset",
    "humanitarian_crisis_onset",
    # Supplementary outcomes (notebooks 14c–14g sources)
    "unrest_binary",
    "fh_status_decline",
    "ethnic_exclusion_any",
    "epr_state_collapse",
    "pts_high",
    "pts_escalation",
    "aut_ep",
]
ID_COLS = ["iso3", "year"]

# Numeric feature columns only (exclude ID and outcome columns)
num_cols = [
    c for c in panel.columns
    if c not in ID_COLS + OUTCOME_COLS
    and pd.api.types.is_numeric_dtype(panel[c])
]

panel_sorted = panel.sort_values(["iso3", "year"]).copy()

# ── Lags (t-1, t-2) ──────────────────────────────────────────────────────────
lag_frames = []
for lag in (1, 2):
    lagged = (
        panel_sorted
        .groupby("iso3")[num_cols]
        .shift(lag)
        .rename(columns={c: f"{c}_lag{lag}" for c in num_cols})
    )
    lag_frames.append(lagged)

# ── Rolling means (3-year, 5-year) — shift(1) ensures no same-year leakage ──
roll_frames = []
for window in (3, 5):
    rolled = (
        panel_sorted
        .groupby("iso3")[num_cols]
        .apply(lambda g: g.shift(1).rolling(window, min_periods=1).mean())
        .reset_index(level=0, drop=True)
        .rename(columns={c: f"{c}_roll{window}y" for c in num_cols})
    )
    roll_frames.append(rolled)

panel_sorted = pd.concat(
    [panel_sorted] + lag_frames + roll_frames,
    axis=1,
)
print(f"Panel after lags + rolling features: {panel_sorted.shape}")

# ── Missingness flags ─────────────────────────────────────────────────────────
miss_rate = panel_sorted[num_cols].isna().mean()
high_miss  = miss_rate[miss_rate > 0.05].index.tolist()
for col in high_miss:
    panel_sorted[f"{col}_missing"] = panel_sorted[col].isna().astype("Int8")

print(f"Missingness flags added for {len(high_miss)} columns (>5% missing)")
print(f"Final panel shape: {panel_sorted.shape}")

## Write outputs to ADLS

Two parquets:
- `feature_matrix.parquet` — full panel (features only, no outcomes)
- `labels.parquet` — iso3 + year + 12 outcome columns (5 core + 7 supplementary)

In [ ]:
# ── Feature matrix (no outcomes) ─────────────────────────────────────────────
feature_cols = [c for c in panel_sorted.columns if c not in OUTCOME_COLS]
df_features  = panel_sorted[feature_cols].reset_index(drop=True)

write_parquet(
    df_features,
    f"processed/feature_matrix/{RUN_DATE}/feature_matrix.parquet",
)

# ── Labels ────────────────────────────────────────────────────────────────────
df_labels = labels.reset_index(drop=True)

write_parquet(
    df_labels,
    f"processed/feature_matrix/{RUN_DATE}/labels.parquet",
)

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("=" * 60)
print("Notebook 14 complete")
print("=" * 60)
print(f"  Feature matrix : {df_features.shape[0]:,} rows × {df_features.shape[1]} columns")
print(f"  Labels         : {df_labels.shape}")
print()
print("Outcome label counts (non-NA):")
for col in OUTCOME_COLS:
    n_valid = df_labels[col].notna().sum()
    n_pos   = (df_labels[col] == 1).sum()
    print(f"  {col:<30} n={n_valid:,}  positives={n_pos:,}  "
          f"rate={n_pos/n_valid:.3f}" if n_valid else f"  {col:<30} NA")
print()
print("ADLS paths:")
print(f"  processed/feature_matrix/{RUN_DATE}/feature_matrix.parquet")
print(f"  processed/feature_matrix/{RUN_DATE}/labels.parquet")